## Single frame metrics

In [1]:
import open3d as o3d
import numpy as np

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


###  Point density & density variance (Nearest neighbor)

In [2]:
def density_metrics(pcd, k=10):
    pcd_tree = o3d.geometry.KDTreeFlann(pcd)
    pts = np.asarray(pcd.points)
    dists = []
    for p in pts:
        [_, idx, d] = pcd_tree.search_knn_vector_3d(p, k + 1)
        dists.append(np.mean(np.sqrt(d[1:])))  # exclude self
    dists = np.array(dists)
    
    print(f"mean_nn_dist_mm: { np.mean(dists)} \n density_variance: {np.var(dists)} \n density_std: {np.std(dists)}")
    return {
        "mean_nn_dist_mm": np.mean(dists),
        "density_variance": np.var(dists),
        "density_std": np.std(dists),
    }

### Normal consistency

In [3]:
def normal_consistency(pcd, k=10):
    pcd.estimate_normals(
        search_param=o3d.geometry.KDTreeSearchParamKNN(knn=30)
    )
    pcd_tree = o3d.geometry.KDTreeFlann(pcd)
    pts = np.asarray(pcd.points)
    normals = np.asarray(pcd.normals)
    scores = []
    for i, p in enumerate(pts):
        [_, idx, _] = pcd_tree.search_knn_vector_3d(p, k + 1)
        neighbours = idx[1:]
        cos_sims = np.abs(normals[neighbours] @ normals[i])
        scores.append(np.mean(cos_sims))
        
    print(f"\n Normal consistency {np.mean(scores)}")
    return np.mean(scores)  # 0–1, higher is smoother

### Local urvature

In [4]:
# Curvature = the smallest eigenvalue of the local covariance matrix, normalized by the sum of eigenvalues to be scale-invariant 
# => Higher curvature means more local surface variation
# PCA based
def local_curvature(pcd, k=20):
    pcd_tree = o3d.geometry.KDTreeFlann(pcd)
    pts = np.asarray(pcd.points)
    curvatures = []
    for p in pts:
        [_, idx, _] = pcd_tree.search_knn_vector_3d(p, k + 1)
        neighbours = pts[idx[1:]]
        cov = np.cov((neighbours - neighbours.mean(axis=0)).T)
        eigvals = np.linalg.eigvalsh(cov)
        eigvals = np.sort(np.abs(eigvals))
        curvature = eigvals[0] / (eigvals.sum() + 1e-8)
        curvatures.append(curvature)
        
    print(f"\nmean_curvature: {np.mean(curvatures)} \n max_curvature: {np.max(curvatures)} \n curvature_std: {np.std(curvatures)}")
    return {
        "mean_curvature": np.mean(curvatures),
        "max_curvature": np.max(curvatures),
        "curvature_std": np.std(curvatures),
    }

## Results

In [7]:
def evaluate(pcd):
    density_metrics(pcd)
    normal_consistency(pcd)

    # INACCURATE FOR POINT CLOUDS
    # local_curvature(pcd)

# print("\n-- All filters")
# pcd = o3d.io.read_point_cloud("../data/filters_exhale2.ply")
# evaluate(pcd)

# print("\n-- No temp filter")
# pcd = o3d.io.read_point_cloud("../data/no_tempfilter1.ply")
# evaluate(pcd)

# print("\n-- Temp filter")
# pcd = o3d.io.read_point_cloud("../data/temp_filter1.ply")
# evaluate(pcd)

print("\n-- reg calibration")
pcd = o3d.io.read_point_cloud("../frame_extraction/results/test_aloys_reg_calibration/pointcloud_00020.ply")
evaluate(pcd)

print("\n-- WITH dyn calibration") 
pcd = o3d.io.read_point_cloud("../output_filtered_20.ply")
evaluate(pcd)


-- reg calibration
mean_nn_dist_mm: 0.00227031104018055 
 density_variance: 6.091250576228248e-07 
 density_std: 0.0007804646421349431

 Normal consistency 0.9937504646846762

-- WITH dyn calibration
mean_nn_dist_mm: 0.004414801073355143 
 density_variance: 8.531462144005851e-07 
 density_std: 0.0009236591440572573

 Normal consistency 0.9970756263491715


In [16]:
def evaluate(pcd):
    print("Results for PC: \n")
    density_metrics(pcd)
    normal_consistency(pcd)
    local_curvature(pcd)

pcd = o3d.io.read_point_cloud("../data/filters_inhale2.ply")
evaluate(pcd)

Results for PC: 

mean_nn_dist_mm: 0.0049426452549092585 
 density_variance: 2.0463439147383252e-06 
 density_std: 0.0014305047762025562

 Normal consistency 0.9947724768585511

mean_curvature: 0.005083672979390296 
 max_curvature: 0.14944944940126945 
 curvature_std: 0.007787400344672492


In [ ]:
import pyrealsense2 as rs
from scipy.linalg import lstsq

def measure_camera_quality(pipeline, n_frames=40, roi_frac=0.3):
    """    
    n_frames: number of frames to average over (Intel uses 40)
    roi_frac: fraction of image to use as central ROI
    """
    depth_intrinsics = pipeline.get_active_profile() \
        .get_stream(rs.stream.depth).as_video_stream_profile().intrinsics
    depth_scale = pipeline.get_active_profile() \
        .get_device().first_depth_sensor().get_depth_scale()

    frames_depth = []
    for _ in range(n_frames):
        fs = pipeline.wait_for_frames()
        df = fs.get_depth_frame()
        frames_depth.append(np.asanyarray(df.get_data()).astype(float) * depth_scale)

    stack = np.stack(frames_depth, axis=0)  # (N, H, W) in metres
    H, W = stack.shape[1], stack.shape[2]

    r0, r1 = int(H * (0.5 - roi_frac/2)), int(H * (0.5 + roi_frac/2))
    c0, c1 = int(W * (0.5 - roi_frac/2)), int(W * (0.5 + roi_frac/2))
    roi = stack[:, r0:r1, c0:c1]

    valid_mask = roi > 0
    fill_rate = valid_mask.mean() * 100  # % — should be >= 99%

    std_n   = np.std(roi, axis=0)
    median_n = np.median(roi, axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(median_n > 0, std_n / median_n, np.nan)
    temporal_noise = np.nanmedian(ratio) * 100  # % — should be <= 1%

    # --- Spatial Noise (RMS from best-fit plane) ---
    # Use median frame to reduce temporal noise influence
    median_frame = np.median(roi, axis=0)
    ys, xs = np.where(median_frame > 0)
    zs = median_frame[ys, xs]

    # Deproject to 3D
    fx, fy = depth_intrinsics.fx, depth_intrinsics.fy
    ppx, ppy = depth_intrinsics.ppx, depth_intrinsics.ppy
    xs_3d = (xs + c0 - ppx) * zs / fx
    ys_3d = (ys + r0 - ppy) * zs / fy
    pts = np.stack([xs_3d, ys_3d, zs], axis=1)

    # Fit plane ax + by + cz = 1 using least squares
    A = pts
    b = np.ones(len(pts))
    coeffs, _, _, _ = lstsq(A, b)
    a, b_c, c = coeffs
    norm = np.sqrt(a**2 + b_c**2 + c**2)

    # Signed distance from each point to the plane
    distances = (pts @ coeffs - 1) / norm
    rms_error_mm = np.sqrt(np.mean(distances**2)) * 1000  # mm — should be <= 2% of distance

    # --- Z-Accuracy (requires known ground truth distance) ---
    measured_dist_m = np.median(zs)  # median measured depth in metres
    # z_accuracy = (measured_dist_m - ground_truth_m) / ground_truth_m * 100
    # Provide your measured ground truth (e.g. from a tape measure) here

    return {
        "fill_rate_%":        round(fill_rate, 2),       # target: >= 99%
        "temporal_noise_%":   round(temporal_noise, 4),  # target: <= 1%
        "spatial_noise_mm":   round(rms_error_mm, 3),    # target: <= 2% of distance
        "measured_dist_m":    round(measured_dist_m, 4), # compare to your tape measure
        "n_valid_pixels":     int(valid_mask.sum()),
    }